# GRAIL Stage 2a reranker — full build-out (Colab Pro+)

Scales the **no-MCS bi-encoder reranker** (gate verdict: GO, val 0.497@15 > generator 0.440) to a publishable headline: larger training, full-val selection + TEST-once + GLORYx-37 cross-method, seeds, ablations.

**Runtime → GPU** (training is fast on GPU; note the *pool generation* is RDKit/CPU-bound — the slow part — so use a Pro+ high-RAM/high-core runtime and let pools cache to Drive so they survive disconnects).

**Before running, put these in your Google Drive** (the dataset + checkpoint are gitignored, not in the repo):
- the GRAIL repo (push the `claude/hungry-pasteur-25d746` branch to GitHub, OR zip the repo to Drive),
- `train.sdf val.sdf test.sdf` + `*_triples_clean.txt` (and `*_triples.txt`),
- the trained generator checkpoint `artifacts/full5000_priors/checkpoints/generator.pt` (+ `filter.pt`),
- `resources/extended_smirks.txt` is in the repo already.

Edit the **CONFIG** cell paths, then Run all.

In [ ]:
# 1) Mount Drive + CONFIG (edit these paths to match your Drive layout)
from google.colab import drive
drive.mount('/content/drive')

GITHUB_URL = ''        # e.g. 'https://github.com/<you>/GRAIL.git'  (leave '' to copy from Drive instead)
BRANCH     = 'claude/hungry-pasteur-25d746'
DRIVE_REPO = '/content/drive/MyDrive/GRAIL'          # used only if GITHUB_URL == '' (a repo copy on Drive)
DRIVE_DATA = '/content/drive/MyDrive/GRAIL_data'      # holds train.sdf/val.sdf/test.sdf + *_triples*.txt
DRIVE_CKPT = '/content/drive/MyDrive/GRAIL_ckpt'      # holds generator.pt (+ filter.pt) of full5000_priors
DRIVE_CACHE= '/content/drive/MyDrive/GRAIL_reranker_cache'  # pools cache (survives disconnects)
print('config set')

In [ ]:
# 2) Get the repo + install
import os, subprocess
if GITHUB_URL:
    subprocess.run(['git','clone','--branch',BRANCH,'--depth','1',GITHUB_URL,'/content/GRAIL'], check=True)
else:
    subprocess.run(['cp','-r',DRIVE_REPO,'/content/GRAIL'], check=True)
%cd /content/GRAIL
!pip -q install 'numpy<2' rdkit torch torch_geometric 2>&1 | tail -2
!pip -q install -e . 2>&1 | tail -2
import numpy, torch; print('numpy', numpy.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 3) Wire Drive data + checkpoint + cache into the repo's expected paths (symlinks)
import os
os.makedirs('grail_metabolism/data', exist_ok=True)
for f in ['train.sdf','val.sdf','test.sdf','train_triples.txt','val_triples.txt','test_triples.txt',
          'train_triples_clean.txt','val_triples_clean.txt','test_triples_clean.txt']:
    src = os.path.join(DRIVE_DATA, f)
    if os.path.exists(src):
        dst = os.path.join('grail_metabolism/data', f)
        if not os.path.exists(dst): os.symlink(src, dst)
os.makedirs('artifacts/full5000_priors/checkpoints', exist_ok=True)
for f in ['generator.pt','filter.pt']:
    src = os.path.join(DRIVE_CKPT, f)
    if os.path.exists(src):
        dst = os.path.join('artifacts/full5000_priors/checkpoints', f)
        if not os.path.exists(dst): os.symlink(src, dst)
# pools cache on Drive (survives runtime disconnects)
os.makedirs(DRIVE_CACHE, exist_ok=True)
if not os.path.exists('artifacts/reranker_gate_cache'):
    os.symlink(DRIVE_CACHE, 'artifacts/reranker_gate_cache')
print('data:', sorted(os.listdir('grail_metabolism/data'))[:6])
print('ckpt:', os.listdir('artifacts/full5000_priors/checkpoints'))
!python -m pytest grail_metabolism/tests/test_reranker.py -q 2>&1 | tail -3

In [ ]:
# 4) SCALE training + multi-seed (no-MCS bi-encoder). Pool-gen is the wall (~5-6s/substrate,
#    CPU); it caches to Drive so a disconnect+rerun resumes. Adjust scale to your time budget.
import subprocess
for seed in [0, 1]:
    subprocess.run(['python','scripts/run_reranker_gate.py','--arch','bi',
                    '--train-substrates','2500','--val-substrates','600',
                    '--top-k','100','--max-pool','80','--epochs','20','--seed',str(seed)], check=True)
    !cp results/reranker_gate_bi.json results/reranker_gate_bi_seed{seed}.json
!echo '--- per-seed val headlines ---'
import json, glob
for f in sorted(glob.glob('results/reranker_gate_bi_seed*.json')):
    h = json.load(open(f))['headline']; print(f, '->', round(h['reranker_recall@15'],3),
          'vs gen', round(h['generator_alone_recall@15'],3), 'oracle', round(h['oracle_recall@15'],3))

In [ ]:
# 5) Cross-method headline: GLORYx-37 (+ clean TEST) under the standardized protocol.
#    reranker_predict.py trains on the cached pools then reranks each set's pools to top-15.
!python scripts/reranker_predict.py --substrates gloryx --out docs/benchmark/data/grail_reranker_gloryx.json
!python scripts/eval_on_gloryx.py --no-grail --no-sygma \
    --extra GRAIL_reranker=docs/benchmark/data/grail_reranker_gloryx.json \
            BioTransformer=docs/benchmark/data/biotransformer_gloryx.json \
            MetaPredictor=docs/benchmark/data/metapredictor_gloryx.json
# clean TEST (touch once for the final report):
!python scripts/reranker_predict.py --substrates test --out results/grail_reranker_test.json

## 6) Ablations — PENDING flag support (add before running)

These quantify each component's contribution. They need flags I will add to `BiEncoderReranker` / `run_reranker_gate.py` after the local peek confirms the direction:
- `--no-rule-prior` (drop the `rule_prior_logits` scalar feature),
- `--no-gen-score` (drop the `gen_score` scalar feature),
- `--rns` (add the k-NN analogue-retrieval site-posterior feature — the originally-designed RNS component).

Once added, run e.g. `run_reranker_gate.py --arch bi ... --no-rule-prior` and compare val recall@15 to the full model. (Cell left as a placeholder so the notebook doesn't fail on missing flags.)

In [ ]:
# 7) Download the results bundle
import shutil, glob
shutil.make_archive('/content/reranker_results','zip','results')
from google.colab import files
files.download('/content/reranker_results.zip')